In [ ]:
import numpy as np

path = "2025_combined_mlar_header.txt"

selected_columns = [
    "state_code",
    "loan_type",
    "action_taken",
    "debt_to_income_ratio"
]

with open(path, "r", encoding="utf-8-sig") as f:
    header = f.readline().strip().split("|")

column_indices = [
    header.index(col) for col in selected_columns
]

extracted_data = np.genfromtxt(
    path,
    delimiter="|",
    skip_header=1,
    usecols=column_indices,
    max_rows=9998,
    dtype=str,
    encoding="utf-8"
)

missing_mask = np.isin(
    np.char.strip(extracted_data),
    ["", "NA", "nan", "NaN"]
)

for i, name in enumerate(selected_columns):
    print(name, "missing:", np.sum(missing_mask[:, i]))

data = extracted_data.copy()
data = data[:, [3, 0, 1, 2]]

dti_missing = np.isin(
    np.char.upper(np.char.strip(data[:, 0].astype(str))),
    ["", "NA", "NAN", "NONE", "NULL"]
)

state_missing = np.isin(
    np.char.upper(np.char.strip(data[:, 1].astype(str))),
    ["", "NA", "NAN", "NONE", "NULL"]
)

valid_rows = ~(state_missing | dti_missing)
final_data = data[valid_rows].copy()

print(final_data)

In [ ]:
sample_data = final_data

city=sample_data[:,1]
unique_city=np.unique(city)
print("unique_city")
# print(unique_city)
city_list=unique_city.tolist()
print(city_list)
state=input("Enter your state:").strip().upper()
while state not in city_list:
    print("Invalid choice. Please try again")
    state=input("Enter your state:").strip().upper()
print("selected state")    
print(state)

loan=sample_data[:,2]
unique_loan=np.unique(loan)
print("unique loan")
# print(unique_loan)
loan_list=unique_loan.tolist()
loan_list= [str(x) for x in loan_list]
# print(loan_list)
loan_type=input("Enter your loan type:").strip()
while loan_type not in loan_list:
    print("Invalid choice. Please try again")
    loan_type=input("Enter your loan type:").strip()
print("loan_type selected:")    
print(int(loan_type))

# action=sample_data[:,3]
# unique_action=np.unique(action)
# print(unique_action)
# action_list=unique_action.tolist()
# action_list= [str(x) for x in loan_list]
# print(action_list)
# action_taken=input("Enter your action taken:").strip()
# while action_taken not in action_list:
#     print("Invalid choice. Please try again")
#     action_taken=input("Enter your action taken:").strip()
# print(int(action_taken))

state_condition=city==state
# print(state_condition)
selected_state=sample_data[state_condition]
# print(selected_state)
# loan_type=int(loan_type)
loan_type_condition=loan==loan_type
# print(loan_type_condition)
selected_loan_type=sample_data[loan_type_condition]
# print(loan_type_condition)
# print(selected_loan_type)
combined=state_condition&loan_type_condition
filtered_data=sample_data[combined]
print(filtered_data)

In [ ]:
def find_dti_group(dti):
    if dti in ["","NA","Exempt"]:
        group=None
    elif dti=="<20%":
        group="<20%"
    elif dti =="20%-<30%":
        group="20%-<30%"
    elif dti=="30%-<36%":
        group="30%-<36%"
    elif dti=="50%-60%":
        group="50%-60%"
    elif dti==">60%":
        group=">60%"
    else:
        dti_number=int(dti)
        if dti_number>=36 and dti_number<50:
            group="36%-<50%"
        else:
            group=None
    return group

In [ ]:
print(records)

In [ ]:
records = filtered_data
records = np.delete(records, 1, axis=1)

# # for record in records:
# #     dti=record[0]
# #     action=record[1]
# #     group=find_dti_group(dti)
# #     # print("DTI:",dti,"Group:",group,"Action:",action)

# # Name of the six DTI groups
# groups=[
#     "<20%",
#     "20%-<30%",
#     "30%-<36%",
#     "36%-<50%",
#     "50%-60%",
#     ">60%"
# ]
# #Start each group's counts at zero
# totals=[0,0,0,0,0,0]
# approved_counts=[0,0,0,0,0,0]
# #Check each application
# for record in records:
#     dti=record[0]
#     action=record[1]
#     group=find_dti_group(dti)

#     if action in ["1","2","3"]:
#         for i in range(len(groups)):
#             if group==groups[i]:
#                 totals[i]+=1
#                 if action in ["1","2"]:
#                     approved_counts[i]+=1

# #Display the counts for each group
# for i in range(len(groups)):
#     print(groups[i],"Total:",totals[i],"Approved:",approved_counts[i])
                
# rates=[]

# for i in range(len(groups)):
#     if totals[i]>0:
#         rate=approved_counts[i]/totals[i]
#         # print(groups[i],"Approved rate",rate*100,"%")
#     else:
#         rate=None
#         # print(groups[i],"No valid applications")
#     rates.append(rate)

def calculate_approval_rates(records):
    groups=["<20%",
           "20%-<30%",
           "30%-<36%",
           "36%-<50%",
           "50%-60%",
           ">60%"]

    totals=[0,0,0,0,0,0]
    approved_counts=[0,0,0,0,0,0]

    #count valid and approved applications
    for record in records:
        dti=record[0]
        action=record[1]
        group=find_dti_group(dti)

        if action in ["1","2","3"]:
            for i in range(len(groups)):
                if group==groups[i]:
                    totals[i]+=1
                    if action in ["1","2"]:
                        approved_counts[i]+=1

    #Calculate approval rates
    rates=[]

    for i in range(len(groups)):
        if totals[i] > 0:
            rate = approved_counts[i]/totals[i]
        else:
            rate = None

        rates.append(rate)

    return groups, totals, approved_counts, rates

groups,totals,approved_counts,rates=calculate_approval_rates(records)
size=np.array(totals)

print("Groups:",groups)
print("Totals:",totals)
print("Approved:",approved_counts)
print("Rates:",rates)
print("Size:",size)

# groups,totals,approved_counts,rates=calculate_approval_rates(check_records)

# size=np.array(totals)

# # print("Totals:",totals)
# # print("Approved:",approved_counts)
# # print("Rates:",rates)
# # print("Size:",size)

# empty_records=[]

# groups,totals,approved_counts,rates=calculate_approval_rates(empty_records)

# # print("Totals:",totals)
# # print("Approved:",approved_counts)
# # print("Rates:",rates)

In [ ]:
# column_names = [
#     "state",
#     "loan_type",
#     "action_taken",
#     "debt_to_income"
# ]

# import numpy as np

# data = extracted_data.copy()

# data = data[:, [3, 0, 1, 2]]

# dti_missing = np.isin(
#     np.char.upper(np.char.strip(data[:, 0].astype(str))),
#     ["", "NA", "NAN", "NONE", "NULL"]
# )

# state_missing = np.isin(
#     np.char.upper(np.char.strip(data[:, 1].astype(str))),
#     ["", "NA", "NAN", "NONE", "NULL"]
# )

# valid_rows = ~(dti_missing | state_missing)

# final_data = data[valid_rows].copy()

# print("Original shape:", data.shape)
# print("DTI missing:", np.sum(dti_missing))
# print("State missing:", np.sum(state_missing))
# print("Deleted rows:", np.sum(~valid_rows))
# print("Final shape:", final_data.shape)

# print("Remaining DTI missing:", np.sum(dti_missing[valid_rows]))
# print("Remaining State missing:", np.sum(state_missing[valid_rows]))

# print(final_data[:10])

# import numpy as np

# column_names = ["DTI", "State", "LoanType", "action_taken"]

# for i, name in enumerate(column_names):
#     values = np.char.upper(
#         np.char.strip(final_data[:, i].astype(str))
#     )

#     missing = np.isin(
#         values,
#         ["", "NA", "NAN", "NONE", "NULL"]
#     )

#     print(name, "missing values:", np.sum(missing))

# print("Final data shape:", final_data.shape)

# import numpy as np

# reordered_data = cleaned_data[:, [3, 0, 1, 2]]

# print(reordered_data[:10])
# print(reordered_data.shape)
# print(type(reordered_data))

# import numpy as np

# column_names = ["DTI", "State", "LoanType", "action_taken"]

# for i, name in enumerate(column_names):
#     values = np.char.upper(
#         np.char.strip(final_data[:, i].astype(str))
#     )

#     missing = np.isin(
#         values,
#         ["", "NA", "NAN", "NONE", "NULL"]
#     )

#     print(name, "missing values:", np.sum(missing))

# print("Final data shape:", final_data.shape)

# total_records = extracted_data.shape[0]
# retained_records = final_data.shape[0]
# excluded_records = total_records - retained_records

# print("Total records read:", total_records)
# print("Excluded records:", excluded_records)
# print("Retained records:", retained_records)

# print("Exclusion rate:", round(excluded_records / total_records * 100, 2), "%")
# print("Retention rate:", round(retained_records / total_records * 100, 2), "%")

# print(data.shape)
# print(data.dtype.names)